In [ ]:
import torch
import os
from pathlib import Path
# change base folder
os.chdir('../')
# Load your model definition and dataset
from models import get_model
from types import SimpleNamespace
import yaml
import matplotlib.pyplot as plt
from flame_model.FLAME import FLAMEModel
from renderer.renderer import Renderer
from pytorch3d.transforms import matrix_to_euler_angles
import matplotlib.animation as animation
import numpy as np
from dataset.data_loader_joint_data_batched import get_dataloaders
from scipy.signal import savgol_filter
from base.baseTrainer import load_state_dict
import torch.nn as nn
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2FeatureExtractor
import soundfile as sf
import glob
import subprocess
import torch.nn.functional as F


device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flame    = FLAMEModel(n_shape=300,n_exp=50).to(device)
renderer = Renderer(render_full_head=True).to(device)

In [ ]:
target_dir = "/mnt/fasttalk/demo/video"
import shutil, os; [shutil.rmtree(os.path.join(target_dir, f)) if os.path.isdir(os.path.join(target_dir, f)) else os.remove(os.path.join(target_dir, f)) for f in os.listdir(target_dir)]


In [ ]:
def load_and_flatten_yaml(config_path):
    """
    Loads the YAML file and flattens the structure so that
    all sub-keys under top-level sections (e.g., DATA, NETWORK, etc.)
    appear in a single dictionary without the top-level keys.
    """
    with open(config_path, 'r') as f:
        full_config = yaml.safe_load(f)

    # Flatten the dict by merging all sub-dicts
    flattened_config = {}
    for top_level_key, sub_dict in full_config.items():
        # sub_dict should itself be a dict of key-value pairs
        if isinstance(sub_dict, dict):
            # Merge each sub-key into flattened_config
            for k, v in sub_dict.items():
                flattened_config[k] = v
        else:
            # In case there's a non-dict top-level key (unlikely but possible)
            flattened_config[top_level_key] = sub_dict

    return SimpleNamespace(**flattened_config)

In [ ]:
# ---- Load model (without DDP for eval) ----
def load_model_for_eval(checkpoint_path,cfg):
    model = get_model(cfg)
    model = model.cuda()
   
    checkpoint = torch.load(checkpoint_path, map_location=lambda storage, loc: storage.cpu())
    load_state_dict(model, checkpoint['state_dict'], strict=False)
    print("=> Loaded checkpoint '{}'".format(checkpoint_path))
    
    model.eval()
    return model

# ---- Load dataset ----
def load_dataset(cfg,test_config):

    dataset = get_dataloaders(cfg,test_config)

    if not test_config:
        train_loader = dataset['train']
        val_loader = dataset['valid']
        return train_loader, val_loader
    else:
        test_loader = dataset['test']
        return test_loader


# ---- Evaluate some samples ----
def evaluate_samples(model, data_loader, num_samples=10):
    for i, (padded_blendshapes, blendshape_mask, padded_audios, audio_mask) in enumerate(data_loader):
        if i >= num_samples:
            break
       
        padded_blendshapes  = padded_blendshapes.cuda(device, non_blocking=True)
        blendshape_mask     = blendshape_mask.cuda(device, non_blocking=True)
        padded_audios       = padded_audios.cuda(device, non_blocking=True)
        audio_mask          = audio_mask.cuda(device, non_blocking=True)

        with torch.no_grad():
            loss, loss_detail, blendshapes_out  = model(padded_blendshapes,blendshape_mask,padded_audios,audio_mask,criterion=nn.MSELoss())
        
        print("loss_blendshapes: ", loss_detail[0].item(), "- loss_reg:", loss_detail[1].item())

        render_comparison(padded_blendshapes.squeeze(), blendshapes_out.squeeze(), i)


def get_vertices_from_blendshapes(expr, gpose, jaw, eyelids):

    # Load the encoded file
    expr_tensor    = expr.to(device)
    gpose_tensor   = gpose.to(device)
    jaw_tensor     = jaw.to(device)
    eyelids_tensor = eyelids.to(device)

    target_shape_tensor = torch.zeros(expr_tensor.shape[0], 300).expand(expr_tensor.shape[0], -1).to(device)

    I = matrix_to_euler_angles(torch.cat([torch.eye(3)[None]], dim=0),"XYZ").to(device)

    eye_r    = I.clone().to(device).squeeze()
    eye_l    = I.clone().to(device).squeeze()
    eyes     = torch.cat([eye_r,eye_l],dim=0).expand(expr_tensor.shape[0], -1).to(device)

    pose = torch.cat([gpose_tensor, jaw_tensor], dim=-1).to(device)

    flame_output_only_shape,_ = flame.forward(shape_params=target_shape_tensor, 
                                              expression_params=expr_tensor, 
                                              pose_params=pose, 
                                              eye_pose_params=eyes)
    return flame_output_only_shape.detach()

# Assumes flame and renderer are already defined and on correct device

def render_comparison(blendshapes_gt, blendshapes_pred, index=None, name=None):
    # ==== Align lengths ====
    min_len = min(blendshapes_gt.shape[0], blendshapes_pred.shape[0])
    blendshapes_gt = blendshapes_gt[:min_len]
    blendshapes_pred = blendshapes_pred[:min_len]
    
    # ==== Split GT and predicted blendshapes ====
    expr_gt     = blendshapes_gt[:, :50]
    gpose_gt    = blendshapes_gt[:, 50:53]
    jaw_gt      = blendshapes_gt[:, 53:56]
    eyelids_gt  = blendshapes_gt[:, 56:]

    expr_pr     = blendshapes_pred[:, :50]
    gpose_pr    = blendshapes_pred[:, 50:53]
    jaw_pr      = blendshapes_pred[:, 53:56]
    eyelids_pr = blendshapes_pred[:, 56:]

    # Apply Savitzky-Golay filter along the time axis (axis=0)
    #gpose_pr_smooth_np = savgol_filter(gpose_pr.detach().cpu().numpy(), window_length=7, polyorder=2, axis=0)
    gpose_pr_smoothed = gpose_pr #torch.from_numpy(gpose_pr_smooth_np).to(gpose_pr.device)

    # ==== Generate vertices ====
    verts_gt = get_vertices_from_blendshapes(expr_gt, gpose_gt, jaw_gt, eyelids_gt) # vertice_gt.reshape(-1,5023,3)
    verts_pr = get_vertices_from_blendshapes(expr_pr, gpose_pr_smoothed, jaw_pr, eyelids_pr) # vertice_pred.reshape(-1,5023,3) 

    # ==== Camera ====
    cam = torch.tensor([5, 0, 0], dtype=torch.float32).unsqueeze(0).to(verts_gt.device)
    cam = cam.expand(verts_gt.shape[0], -1)

    # ==== Render both sequences ====
    frames_gt = renderer.forward(verts_gt, cam)['rendered_img']         # [T, 3, H, W]
    frames_pr = renderer.forward(verts_pr, cam)['rendered_img']         # [T, 3, H, W]

    # ==== Prepare output folder ====
    os.makedirs("demo/video", exist_ok=True)
    if index is not None:
        video_file = f"demo/video/sample_{index:03d}.mp4"
    else:
        video_file = f"demo/video/{name}.mp4"

    # ==== Create animation ====
    def update(frame_idx, gt_seq, pr_seq, axes):
        gt = gt_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        pr = pr_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        combined = np.concatenate([(gt * 255).astype(np.uint8), (pr * 255).astype(np.uint8)], axis=1)
        axes.clear()
        axes.imshow(combined)
        axes.axis("off")

    fig, ax = plt.subplots(figsize=(10, 5))
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frames_gt.shape[0],
        fargs=(frames_gt, frames_pr, ax),
        interval=100
    )
    ani.save(video_file, writer='ffmpeg', fps=25)
    plt.close(fig)

def render(blendshapes_pred, index=None, name=None, audio_dir=None):
    # ==== Split GT and predicted blendshapes ====
    expr_pr     = blendshapes_pred[:, :50]
    gpose_pr    = blendshapes_pred[:, 50:53] 
    jaw_pr      = blendshapes_pred[:, 53:56]
    eyelids_pr  = blendshapes_pred[:, 56:]

    # Apply Savitzky-Golay filter along the time axis (axis=0)
    #gpose_pr_smooth_np = savgol_filter(gpose_pr.detach().cpu().numpy(), window_length=7, polyorder=2, axis=0)
    gpose_pr_smoothed = gpose_pr #torch.from_numpy(gpose_pr_smooth_np).to(gpose_pr.device)

    # ==== Generate vertices ====
    verts_pr = get_vertices_from_blendshapes(expr_pr, gpose_pr_smoothed, jaw_pr, eyelids_pr) # vertice_pred.reshape(-1,5023,3) 

    # ==== Camera ====
    cam = torch.tensor([5, 0, 0], dtype=torch.float32).unsqueeze(0).to(verts_pr.device)
    cam = cam.expand(verts_pr.shape[0], -1)

    # ==== Render both sequences ====
    frames_pr = renderer.forward(verts_pr, cam)['rendered_img']         # [T, 3, H, W]

    # ==== Prepare output folder ====
    os.makedirs("demo/video", exist_ok=True)
    if index is not None:
        video_file = f"demo/video/sample_{index:03d}.mp4"
    else:
        video_file = f"demo/video/{name}.mp4"

    # ==== Create animation ====
    def update(frame_idx, pr_seq, axes):
        pr = pr_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        scaled = (pr * 255).astype(np.uint8)
        axes.clear()
        axes.imshow(scaled)
        axes.axis("off")

    fig, ax = plt.subplots(figsize=(5, 5))
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frames_pr.shape[0],
        fargs=(frames_pr, ax),
        interval=100
    )
    ani.save(video_file, writer='ffmpeg', fps=25)
    plt.close(fig)

    # =============== Add audio to the video ===============
    
    # Add audio to the video
    audio_file = f'{audio_dir}/{name}.wav'
    output_with_audio = f'demo/video/{name}_with_audio.mp4'
    if os.path.exists(audio_file):
        cmd = f'ffmpeg -y -i {video_file} -i {audio_file} -c:v copy -c:a aac -strict experimental {output_with_audio}'
        subprocess.run(cmd, shell=True)
        print(f"Video with audio saved as {output_with_audio}")
    else:
        print(f"Audio file {audio_file} not found")

In [ ]:
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

global cfg

#cfg = load_and_flatten_yaml("config/joint_data/stage2interactive.yaml")
cfg = load_and_flatten_yaml("config/talkinghead-1kh/stage2.yaml")

cfg.batch_size = 1

In [ ]:
#cfg.vqvae_pretrained_path = "/mnt/fasttalk/checkpoints_styled_300ep/model_s1.pth.tar"

In [ ]:

checkpoint_path = "/mnt/fasttalk/logs/talkinghead/talkinghead-s2/model_100/model.pth.tar" #"/mnt/fasttalk/logs/joint_data/joint_data_finetune_s2/model_50/model.pth.tar" #"/root/Projects/fasttalk/logs/joint_data/joint_data_custom_s2/model_95/model.pth.tar"
model = load_model_for_eval(checkpoint_path,cfg)


In [ ]:
#target_style_neutral = torch.load("/mnt/fasttalk/demo/new_styles/happy_style.pt",map_location=device)
#target_style_neutral.shape
#target_style = target_style_neutral

In [ ]:
#target_style.shape

In [ ]:
#style = np.load("/mnt/Datasets/expressive_ft/synthetic_dataset/npz/video_015.npz")
style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_CatherineCortezMasto_000.npz")
#style = np.load("/mnt/Datasets/ARTalk_data_subset/npz/WDA_CatherineCortezMasto_000_synth.npz")

#style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_TammyBaldwin0_000.npz")
#style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_ChrisCoons_000.npz")
#style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_RichardBlumenthal_000.npz")
#style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_GerryConnolly_000.npz")
#style = np.load("/mnt/Datasets/ARTalk_data/npz/WDA_AdamSmith_000.npz")

#np.load("/mnt/Datasets/expressive_ft/npz/video_015.npz")# np.load("/mnt/Datasets/ARTalk_data/npz/0Uzp8S5tCDM2.npz")
exp = style["exp"].reshape(-1, 50)
jaw = style["jaw"].reshape(-1, 3)
gpose = style["gpose"].reshape(-1, 3)


#gpose = savgol_filter(gpose, window_length=7, polyorder=2, axis=0) 

# eyelids: all ones
eyelids = np.ones((exp.shape[0], 2), dtype=np.float32)

# concat to [T,58]
concat = np.concatenate([exp, gpose, jaw, eyelids], axis=1)

# pytorch tensor
target_style_expresive = torch.from_numpy(concat).float()          # [T,58]
target_style_expresive = target_style_expresive.unsqueeze(0).to(device) # [1,T,58]
#target_style_expresive = target_style_expresive[:,:100,:]
#target_style_expresive = target_style_expresive[:,:500,:]
target_style_expresive = target_style_expresive[:,100:300,:] #catherine

target_style_expresive.shape

In [ ]:
target_style = target_style_expresive.clone()
#target_style[:, :, :50] *= 1.5

# Style pose ablation

This section keeps the same audio and mostly the same style clip, then changes only the style `gpose` channels (`50:53`) to check whether unstable neck motion is coming through the style path.

In [ ]:
def _odd_savgol_window(length, desired=21):
    window = min(desired, length if length % 2 == 1 else length - 1)
    return max(window, 3)


def smooth_style_gpose(style_tensor, window_length=21, polyorder=2):
    style_variant = style_tensor.clone()
    length = style_variant.shape[1]
    window = _odd_savgol_window(length, window_length)
    if window <= polyorder:
        return style_variant

    gpose_np = style_variant[0, :, 50:53].detach().cpu().numpy()
    gpose_smooth = savgol_filter(gpose_np, window_length=window, polyorder=polyorder, axis=0)
    style_variant[0, :, 50:53] = torch.from_numpy(gpose_smooth).to(
        device=style_variant.device,
        dtype=style_variant.dtype,
    )
    return style_variant


def zero_style_gpose(style_tensor):
    style_variant = style_tensor.clone()
    style_variant[:, :, 50:53] = 0.0
    return style_variant


def mean_style_gpose(style_tensor):
    style_variant = style_tensor.clone()
    style_variant[:, :, 50:53] = style_variant[:, :, 50:53].mean(dim=1, keepdim=True)
    return style_variant


def build_style_ablation_variants(style_tensor):
    return {
        "style_original": style_tensor.clone(),
        "style_smooth_gpose": smooth_style_gpose(style_tensor),
        "style_zero_gpose_keep_expr_jaw": zero_style_gpose(style_tensor),
        "style_mean_gpose_keep_expr_jaw": mean_style_gpose(style_tensor),
        "no_target_style": None,
    }


def pose_stats(blendshapes_tensor):
    blendshapes = blendshapes_tensor.detach()
    if blendshapes.dim() == 3:
        blendshapes = blendshapes.squeeze(0)
    gpose = blendshapes[:, 50:53]
    if gpose.shape[0] < 2:
        return {
            "gpose_abs_mean": gpose.abs().mean().item(),
            "gpose_std": gpose.std().item(),
            "gpose_vel_mean": 0.0,
            "gpose_vel_p95": 0.0,
            "gpose_acc_mean": 0.0,
        }

    velocity = gpose[1:] - gpose[:-1]
    velocity_norm = torch.linalg.norm(velocity, dim=-1)
    if gpose.shape[0] >= 3:
        acceleration = velocity[1:] - velocity[:-1]
        acceleration_norm = torch.linalg.norm(acceleration, dim=-1)
        acceleration_mean = acceleration_norm.mean().item()
    else:
        acceleration_mean = 0.0

    return {
        "gpose_abs_mean": gpose.abs().mean().item(),
        "gpose_std": gpose.std().item(),
        "gpose_vel_mean": velocity_norm.mean().item(),
        "gpose_vel_p95": torch.quantile(velocity_norm, 0.95).item(),
        "gpose_acc_mean": acceleration_mean,
    }


style_ablation_variants = build_style_ablation_variants(target_style)
for variant_name, variant_style in style_ablation_variants.items():
    if variant_style is None:
        print(variant_name, "target_style=None")
    else:
        print(variant_name, pose_stats(variant_style))

# Style embedding sensitivity test

This checks whether changing raw style pose changes the model's internal style embeddings, and whether those changes actually reach the generated output pose.

In [ ]:
import pandas as pd


def _as_table(rows):
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(list(rows[0].keys())[:1]).reset_index(drop=True)


def _pose_velocity(gpose_tensor):
    if gpose_tensor.shape[0] < 2:
        return torch.zeros((0,), device=gpose_tensor.device, dtype=gpose_tensor.dtype)
    return torch.linalg.norm(gpose_tensor[1:] - gpose_tensor[:-1], dim=-1)


def _tensor_pose_stats(name, tensor):
    if tensor.dim() == 3:
        tensor = tensor.squeeze(0)
    gpose = tensor[:, 50:53]
    velocity = _pose_velocity(gpose)
    return {
        "variant": name,
        "gpose_abs_mean": gpose.abs().mean().item(),
        "gpose_std": gpose.std().item(),
        "gpose_vel_mean": velocity.mean().item() if velocity.numel() else 0.0,
        "gpose_vel_p95": torch.quantile(velocity, 0.95).item() if velocity.numel() else 0.0,
    }


def _style_embedding_summary(style_variants, target_len=None):
    if target_len is None:
        if "blend_np" in globals():
            target_len = blend_np.shape[0]
        elif "target_style" in globals():
            target_len = target_style.shape[1]
        else:
            target_len = 200

    model.eval()
    embeddings = {}
    pooled = {}
    rows = []

    with torch.no_grad():
        for name, variant_style in style_variants.items():
            if variant_style is None:
                continue
            style_seq = model._style_seq_from_target(variant_style, target_len)
            embeddings[name] = style_seq.detach()
            pooled[name] = F.normalize(style_seq.mean(dim=1), dim=-1).detach()

            rows.append({
                "variant": name,
                "target_len": target_len,
                "emb_mean_abs": style_seq.abs().mean().item(),
                "emb_std": style_seq.std().item(),
                "emb_frame_delta_mean": (style_seq[:, 1:] - style_seq[:, :-1]).norm(dim=-1).mean().item() if style_seq.shape[1] > 1 else 0.0,
            })

    pair_rows = []
    ref_name = "style_original" if "style_original" in embeddings else next(iter(embeddings))
    ref_emb = embeddings[ref_name]
    ref_pool = pooled[ref_name]
    for name, emb in embeddings.items():
        min_len = min(ref_emb.shape[1], emb.shape[1])
        frame_l2 = (emb[:, :min_len] - ref_emb[:, :min_len]).norm(dim=-1)
        pooled_cos = F.cosine_similarity(ref_pool, pooled[name], dim=-1).mean().item()
        pair_rows.append({
            "variant": name,
            "ref": ref_name,
            "embedding_frame_l2_mean_vs_ref": frame_l2.mean().item(),
            "embedding_frame_l2_p95_vs_ref": torch.quantile(frame_l2.flatten(), 0.95).item(),
            "pooled_cosine_vs_ref": pooled_cos,
        })

    return pd.DataFrame(rows), pd.DataFrame(pair_rows), embeddings


def _saved_output_pose_summary(video_dir="demo/video"):
    if "results" in globals() and results:
        audio_names = sorted({row["audio"] for row in results})
    elif "wav_files" in globals():
        audio_names = [os.path.splitext(os.path.basename(path))[0] for path in wav_files]
    else:
        audio_names = []

    output_rows = []
    pair_rows = []
    variant_names = list(style_ablation_variants.keys()) if "style_ablation_variants" in globals() else []

    for audio_name in audio_names:
        loaded = {}
        for variant_name in variant_names:
            npz_path = os.path.join(video_dir, f"{audio_name}_{variant_name}.npz")
            if not os.path.exists(npz_path):
                continue
            data = np.load(npz_path)
            pose_key = "pose" if "pose" in data.files else "gpose"
            gpose = torch.from_numpy(data[pose_key]).float()
            loaded[variant_name] = gpose
            velocity = _pose_velocity(gpose)
            output_rows.append({
                "audio": audio_name,
                "variant": variant_name,
                "out_gpose_abs_mean": gpose.abs().mean().item(),
                "out_gpose_std": gpose.std().item(),
                "out_gpose_vel_mean": velocity.mean().item() if velocity.numel() else 0.0,
                "out_gpose_vel_p95": torch.quantile(velocity, 0.95).item() if velocity.numel() else 0.0,
            })

        if "style_original" in loaded:
            ref = loaded["style_original"]
            for variant_name, gpose in loaded.items():
                min_len = min(ref.shape[0], gpose.shape[0])
                diff = gpose[:min_len] - ref[:min_len]
                ref_vel = _pose_velocity(ref[:min_len])
                vel = _pose_velocity(gpose[:min_len])
                vel_diff = vel - ref_vel if vel.shape == ref_vel.shape else torch.zeros((0,))
                pair_rows.append({
                    "audio": audio_name,
                    "variant": variant_name,
                    "out_gpose_l2_mean_vs_original": diff.norm(dim=-1).mean().item(),
                    "out_gpose_l2_p95_vs_original": torch.quantile(diff.norm(dim=-1), 0.95).item(),
                    "out_gpose_vel_absdiff_mean_vs_original": vel_diff.abs().mean().item() if vel_diff.numel() else 0.0,
                })

    return pd.DataFrame(output_rows), pd.DataFrame(pair_rows)


# 1) Raw style pose differences across ablations.
raw_style_pose_table = pd.DataFrame([
    _tensor_pose_stats(name, variant_style)
    for name, variant_style in style_ablation_variants.items()
    if variant_style is not None
])

# 2) Internal style embedding differences after model._style_seq_from_target(...).
style_embedding_table, style_embedding_vs_original, style_embeddings = _style_embedding_summary(style_ablation_variants)

# 3) Generated output pose differences, if ablation .npz files have already been saved.
output_pose_table, output_pose_vs_original = _saved_output_pose_summary()

print("Raw style pose stats")
display(raw_style_pose_table)

print("Style embedding stats")
display(style_embedding_table)

print("Style embedding distance vs original")
display(style_embedding_vs_original)

print("Saved output pose stats")
display(output_pose_table)

print("Saved output pose distance vs original")
display(output_pose_vs_original)

print("Interpretation guide:")
print("- Large raw pose differences + tiny embedding differences => style encoder/autoencoder discards pose.")
print("- Large embedding differences + tiny output differences => decoder ignores pose-related style changes.")
print("- Tiny output differences across all style-pose variants + big no-style difference => style is active, but not as direct pose control.")

In [ ]:
def render_ablation_output(blendshapes_pred, name, audio_file=None):
    expr_pr = blendshapes_pred[:, :50]
    gpose_pr = blendshapes_pred[:, 50:53]
    jaw_pr = blendshapes_pred[:, 53:56]
    eyelids_pr = blendshapes_pred[:, 56:]

    verts_pr = get_vertices_from_blendshapes(expr_pr, gpose_pr, jaw_pr, eyelids_pr)
    cam = torch.tensor([5, 0, 0], dtype=torch.float32).unsqueeze(0).to(verts_pr.device)
    cam = cam.expand(verts_pr.shape[0], -1)
    frames_pr = renderer.forward(verts_pr, cam)["rendered_img"]

    os.makedirs("demo/video", exist_ok=True)
    video_file = f"demo/video/{name}.mp4"

    def update(frame_idx, pr_seq, axes):
        pr = pr_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        axes.clear()
        axes.imshow((pr * 255).astype(np.uint8))
        axes.axis("off")

    fig, ax = plt.subplots(figsize=(5, 5))
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frames_pr.shape[0],
        fargs=(frames_pr, ax),
        interval=100,
    )
    ani.save(video_file, writer="ffmpeg", fps=25)
    plt.close(fig)

    if audio_file is not None and os.path.exists(audio_file):
        output_with_audio = f"demo/video/{name}_with_audio.mp4"
        cmd = [
            "ffmpeg",
            "-y",
            "-i",
            video_file,
            "-i",
            audio_file,
            "-c:v",
            "copy",
            "-c:a",
            "aac",
            "-strict",
            "experimental",
            output_with_audio,
        ]
        subprocess.run(cmd, check=False)
        print(f"Video with audio saved as {output_with_audio}")
    else:
        print(f"Video saved as {video_file}")

In [ ]:
# raw audio style-pose ablation test
# Set MAX_AUDIO_FILES = None to run every wav in the folder.
audio_dir = "/mnt/Datasets/expressive_ft/synthetic_dataset/wav" # "/mnt/fasttalk/demo/input" #
MAX_AUDIO_FILES = 3
USE_QUANTIZER = False

processor = Wav2Vec2FeatureExtractor.from_pretrained(cfg.wav2vec2model_path)
print(cfg.wav2vec2model_path)

wav_files = sorted(glob.glob(os.path.join(audio_dir, "*.wav")))
if MAX_AUDIO_FILES is not None:
    wav_files = wav_files[:MAX_AUDIO_FILES]

results = []
for wav_file in wav_files:
    print("Generating ablations for {}...".format(wav_file))
    test_name = os.path.splitext(os.path.basename(wav_file))[0]
    speech_array, _ = librosa.load(wav_file, sr=16000)

    audio_feature = np.squeeze(processor(speech_array, sampling_rate=16000).input_values)
    audio_feature = np.reshape(audio_feature, (-1, audio_feature.shape[0]))
    audio_feature = torch.FloatTensor(audio_feature).to(device=device)

    for variant_name, variant_style in style_ablation_variants.items():
        output_name = f"{test_name}_{variant_name}"
        print("  variant:", variant_name)

        with torch.no_grad():
            if USE_QUANTIZER:
                blendshapes_out = model.predict(audio_feature, target_style=variant_style)
            else:
                blendshapes_out = model.predict_no_quantizer(audio_feature, target_style=variant_style)

        blend_np = blendshapes_out.squeeze(0).detach().cpu().numpy()
        np.savez(
            os.path.join(target_dir, f"{output_name}.npz"),
            exp=blend_np[:, :50],
            pose=blend_np[:, 50:53],
            jaw=blend_np[:, 53:56],
            eyebrows=blend_np[:, 56:58],
        )

        stats = pose_stats(blendshapes_out)
        stats["audio"] = test_name
        stats["variant"] = variant_name
        results.append(stats)
        print(stats)

        render_ablation_output(blendshapes_out.squeeze(), name=output_name, audio_file=wav_file)

results

In [ ]:
def _ffmpeg_drawtext_escape(text):
    return str(text).replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")


def _find_drawtext_fontfile():
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
    ]
    for font_path in candidates:
        if os.path.exists(font_path):
            return font_path
    return None


def _pretty_variant_title(variant_name):
    titles = {
        "style_original": "Original style",
        "style_smooth_gpose": "Smoothed style pose",
        "style_zero_gpose_keep_expr_jaw": "Zero style pose",
        "style_mean_gpose_keep_expr_jaw": "Mean style pose",
        "no_target_style": "No target style",
    }
    return titles.get(variant_name, variant_name.replace("_", " ").title())


def _run_ffmpeg(cmd):
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        stderr_tail = "\n".join(proc.stderr.splitlines()[-40:])
        stdout_tail = "\n".join(proc.stdout.splitlines()[-20:])
        print("ffmpeg failed. stderr tail:")
        print(stderr_tail)
        if stdout_tail:
            print("ffmpeg stdout tail:")
            print(stdout_tail)
        raise RuntimeError(f"ffmpeg failed with exit code {proc.returncode}")
    return proc


def concat_ablation_cases_for_audio(test_name, audio_file=None, variant_names=None, video_dir="demo/video"):
    if variant_names is None:
        if "style_ablation_variants" in globals():
            variant_names = list(style_ablation_variants.keys())
        else:
            pattern = os.path.join(video_dir, f"{test_name}_*.mp4")
            variant_names = []
            for path in sorted(glob.glob(pattern)):
                stem = os.path.splitext(os.path.basename(path))[0]
                if stem.endswith("_with_audio"):
                    continue
                variant_names.append(stem.replace(f"{test_name}_", "", 1))

    input_videos = []
    kept_variants = []
    for variant_name in variant_names:
        video_path = os.path.join(video_dir, f"{test_name}_{variant_name}.mp4")
        if os.path.exists(video_path):
            input_videos.append(video_path)
            kept_variants.append(variant_name)
        else:
            print(f"Skipping missing video: {video_path}")

    if not input_videos:
        raise FileNotFoundError(f"No ablation videos found for {test_name} in {video_dir}")

    output_path = os.path.join(video_dir, f"{test_name}_all_style_ablation_cases_with_audio.mp4")
    inputs = []
    for video_path in input_videos:
        inputs.extend(["-i", video_path])

    audio_input_index = None
    if audio_file is not None and os.path.exists(audio_file):
        audio_input_index = len(input_videos)
        inputs.extend(["-i", audio_file])

    fontfile = _find_drawtext_fontfile()
    if fontfile is None:
        print("No drawtext font found; concatenating without titles.")

    filter_parts = []
    for idx, variant_name in enumerate(kept_variants):
        video_filter = (
            f"[{idx}:v]scale=512:512:force_original_aspect_ratio=decrease,"
            f"pad=512:512:(ow-iw)/2:(oh-ih)/2,setsar=1"
        )
        if fontfile is not None:
            title = _ffmpeg_drawtext_escape(_pretty_variant_title(variant_name))
            video_filter += (
                f",drawtext=fontfile={fontfile}:text='{title}':"
                f"x=16:y=16:fontsize=26:fontcolor=white:"
                f"box=1:boxcolor=black@0.55:boxborderw=8"
            )
        filter_parts.append(f"{video_filter}[v{idx}]")

    filter_parts.append("".join(f"[v{idx}]" for idx in range(len(input_videos))) + f"hstack=inputs={len(input_videos)}[vout]")
    filter_complex = ";".join(filter_parts)

    cmd = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[vout]"]
    if audio_input_index is not None:
        cmd.extend(["-map", f"{audio_input_index}:a", "-c:a", "aac", "-shortest"])
    cmd.extend(["-c:v", "libx264", "-pix_fmt", "yuv420p", output_path])

    print("Concatenating variants:", kept_variants)
    _run_ffmpeg(cmd)
    print("Saved:", output_path)
    return output_path


# Concatenate every ablation case for each generated audio sample.
concat_outputs = []
if "results" in globals() and results:
    audio_names = sorted({row["audio"] for row in results})
else:
    audio_names = [os.path.splitext(os.path.basename(path))[0] for path in wav_files]

for audio_name in audio_names:
    matching_wavs = [path for path in wav_files if os.path.splitext(os.path.basename(path))[0] == audio_name]
    audio_file = matching_wavs[0] if matching_wavs else None
    concat_outputs.append(concat_ablation_cases_for_audio(audio_name, audio_file=audio_file))

concat_outputs